(sec-10)=
# 10. Bibliografía

Se incluyen solo las referencias citadas en el informe, con su DOI cuando lo tienen; los metadatos de las referencias nuevas de este entregable se consultaron en OpenAlex (`literatura/referencias_e2.json`). La misma lista se exporta en formato BibTeX, con los mismos DOI, en `entregables/11_literatura/referencias.bib`.

In [78]:
import re
import unicodedata

REFS_E2 = json.loads((LIT2 / "referencias_e2.json").read_text(encoding="utf-8"))
# Año y fuente de publicación cuando la versión citada es la de congreso y el DOI es el de arXiv
CONGRESO = {"10.48550/arxiv.1706.03762": (2017, "Advances in Neural Information Processing Systems 30"), "10.48550/arxiv.1612.01474": (2017, "Advances in Neural Information Processing Systems 30"),
            "10.48550/arxiv.1506.02142": (2016, "International Conference on Machine Learning"), "10.48550/arxiv.1703.01365": (2017, "International Conference on Machine Learning"),
            "10.48550/arxiv.1706.04599": (2017, "International Conference on Machine Learning"), "10.48550/arxiv.2004.11362": (2020, "Advances in Neural Information Processing Systems 33"),
            "10.48550/arxiv.2010.01028": (2020, "Advances in Neural Information Processing Systems 33"), "10.48550/arxiv.2106.11959": (2021, "Advances in Neural Information Processing Systems 34")}
def paginas_fmt(paginas):
    # Un artículo de una sola página o con número de artículo se escribe sin rango
    partes = [x.strip() for x in str(paginas).replace("–", "-").split("-") if x.strip()]
    return partes[0] if len(partes) == 2 and partes[0] == partes[1] else "–".join(partes)
def formato_ref(doi, m):
    autores = [partes_nombre(a) for a in m["autores"]]
    lista = [autor_apa(a, n) for a, n in autores]
    texto_aut = lista[0] if len(lista) == 1 else ", ".join(lista[:-1]) + " y " + lista[-1] if len(lista) <= 3 else ", ".join(lista[:3]) + ", et al."
    anio, fuente = CONGRESO.get(doi, (m["anio"], m["fuente"]))
    titulo = m["titulo"].replace("$n$", "n")
    if doi in CONGRESO or m.get("tipo") in ("proceedings-article", "conference-paper") or "Conference" in (fuente or "") or "Proceedings" in (fuente or ""):
        detalle = f"En *{fuente}*."
    elif m.get("tipo") == "book":
        titulo = f"*{titulo}*" + (" (3.ª ed.)" if doi == "10.1002/9781118548387" else "")
        detalle = "Wiley."
    else:
        detalle = f"*{fuente}" + (f", {m['volumen']}*" if m.get("volumen") else "*") + (f"({m['numero']})" if m.get("numero") else "") + (f", {paginas_fmt(m['paginas'])}." if m.get("paginas") else ".")
    return f"- {texto_aut} ({anio}). {titulo}. {detalle} <https://doi.org/{doi}>"
SIN_DOI = [
    "- Benavoli, A., Corani, G., Demšar, J. y Zaffalon, M. (2017). Time for a change: A tutorial for comparing multiple classifiers through Bayesian analysis. *Journal of Machine Learning Research, 18*(77), 1–36.",
    "- Cohen, J. (1988). *Statistical power analysis for the behavioral sciences* (2.ª ed.). Lawrence Erlbaum Associates.",
    "- DeLong, E. R., DeLong, D. M. y Clarke-Pearson, D. L. (1988). Comparing the areas under two or more correlated receiver operating characteristic curves: A nonparametric approach. *Biometrics, 44*(3), 837–845.",
    "- Demšar, J. (2006). Statistical comparisons of classifiers over multiple data sets. *Journal of Machine Learning Research, 7*, 1–30.",
    "- Efron, B. y Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall.",
    "- Friedman, M. (1937). The use of ranks to avoid the assumption of normality implicit in the analysis of variance. *Journal of the American Statistical Association, 32*(200), 675–701.",
    "- Geirhos, R., Jacobsen, J.-H., Michaelis, C., Zemel, R., Brendel, W., Bethge, M. y Wichmann, F. A. (2020). Shortcut learning in deep neural networks. *Nature Machine Intelligence, 2*(11), 665–673.",
    "- Gretton, A., Borgwardt, K. M., Rasch, M. J., Schölkopf, B. y Smola, A. (2012). A kernel two-sample test. *Journal of Machine Learning Research, 13*, 723–773.",
    "- Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics, 6*(2), 65–70.",
    "- Kingma, D. P. y Ba, J. (2015). Adam: A method for stochastic optimization. En *International Conference on Learning Representations*. <https://arxiv.org/abs/1412.6980>",
    "- Lopez-Paz, D. y Oquab, M. (2017). Revisiting classifier two-sample tests. En *International Conference on Learning Representations*.",
    "- McNemar, Q. (1947). Note on the sampling error of the difference between correlated proportions or percentages. *Psychometrika, 12*(2), 153–157.",
    "- Nadeau, C. y Bengio, Y. (2003). Inference for the generalization error. *Machine Learning, 52*(3), 239–281.",
    "- Page, M. J., McKenzie, J. E., Bossuyt, P. M., et al. (2021). The PRISMA 2020 statement: An updated guideline for reporting systematic reviews. *BMJ, 372*, n71. <https://doi.org/10.1136/bmj.n71>",
    "- Schuirmann, D. J. (1987). A comparison of the two one-sided tests procedure and the power approach for assessing the equivalence of average bioavailability. *Journal of Pharmacokinetics and Biopharmaceutics, 15*(6), 657–680.",
    "- SCImago (2026). *SJR: SCImago Journal & Country Rank* [Conjunto de datos]. <https://www.scimagojr.com>",
    "- Shapiro, S. S. y Wilk, M. B. (1965). An analysis of variance test for normality (complete samples). *Biometrika, 52*(3/4), 591–611.",
    "- Wilcoxon, F. (1945). Individual comparisons by ranking methods. *Biometrics Bulletin, 1*(6), 80–83.",
    "- Zeng, Q. (2025). *UAV--IDS2025* [Código fuente]. GitHub. <https://github.com/QingliZeng/UAV--IDS2025>",
    "- Zeng, Q., Bashir, A. y Nait-Abdesselam, F. (2025). *UAVIDS-2025* [Conjunto de datos]. Zenodo. <https://doi.org/10.5281/zenodo.15336998>",
]
FUENTES = {"International journal of intelligent engineering and systems": "International Journal of Intelligent Engineering and Systems",
           "Concurrency and Computation Practice and Experience": "Concurrency and Computation: Practice and Experience", "arXiv (Cornell University)": "arXiv"}
def referencia_estudio(doi):
    m = METADATOS[doi]
    autores = [autor_apa(a, n) for a, n in map(partes_nombre, m["autores"])]
    lista = autores[0] if len(autores) == 1 else ", ".join(autores[:-1]) + " y " + autores[-1] if len(autores) <= 3 else ", ".join(autores[:3]) + ", et al."
    titulo = m["titulo"].capitalize() if m["titulo"].isupper() else m["titulo"]
    fuente = m["fuente"].title() if (m.get("fuente") or "").isupper() else m.get("fuente")
    fuente = FUENTES.get(fuente, fuente)
    paginas = m.get("paginas") or ""
    if m.get("tipo") == "conference-paper":
        detalle = f"En *{fuente}*" + (f" (pp. {paginas_fmt(paginas)})." if paginas else ".")
    else:
        detalle = f"*{fuente}" + (f", {m['volumen']}*" if m.get("volumen") else "*") + (f"({m['numero']})" if m.get("numero") else "") + (f", {paginas_fmt(paginas)}." if paginas else ".")
    enlace_ = m.get("url") if doi.startswith("sin-doi") else f"https://doi.org/{doi}"
    return f"- {lista} ({m['anio']}{SUFIJO.get(doi, '')}). {titulo}. {detalle} <{enlace_}>"
BIBLIOGRAFIA = sorted([referencia_estudio(e["doi"]) for e in EXT] + [formato_ref(d, m) for d, m in REFS_E2.items()] + SIN_DOI, key=clave_orden)
interpretar("\n".join(BIBLIOGRAFIA))

# Misma bibliografía en BibTeX (sección 8 del enunciado), generada de los mismos metadatos
def bib_texto(t):
    return str(t).replace("\\", "").replace("&", r"\&").replace("%", r"\%").replace("_", r"\_").replace("#", r"\#").replace("$n$", "n")
def bib_entrada(tipo, apellido, anio, titulo, campos, usadas):
    base = "".join(c for c in unicodedata.normalize("NFKD", apellido.lower()) if c.isalnum() and not unicodedata.combining(c))
    palabra = next((p for p in re.findall(r"[a-z0-9]+", clave_orden(titulo)) if len(p) > 3), "ref")
    clave = f"{base}{anio}{palabra}"
    while clave in usadas:
        clave += "x"
    usadas.add(clave)
    lineas = [f"  title = {{{{{bib_texto(titulo)}}}}}", f"  year = {{{anio}}}"] + [f"  {k} = {{{bib_texto(v) if k not in ('doi', 'url') else v}}}" for k, v in campos.items() if v]
    return f"@{tipo}{{{clave},\n" + ",\n".join(lineas) + "\n}"
def bib_autores(nombres):
    return " and ".join(f"{a}, {n}" if n.strip() else a for a, n in map(partes_nombre, nombres))
usadas_bib, ENTRADAS_BIB = set(), []
for e in EXT:
    doi = e["doi"]
    m = METADATOS[doi]
    fuente = m["fuente"].title() if (m.get("fuente") or "").isupper() else m.get("fuente")
    fuente = FUENTES.get(fuente, fuente)
    titulo = m["titulo"].capitalize() if m["titulo"].isupper() else m["titulo"]
    enlace_ = {"url": m.get("url")} if doi.startswith("sin-doi") else {"doi": doi}
    if m.get("tipo") == "conference-paper":
        campos = {"author": bib_autores(m["autores"]), "booktitle": fuente, "pages": paginas_fmt(m.get("paginas") or "").replace("–", "--"), **enlace_}
        ENTRADAS_BIB.append(bib_entrada("inproceedings", partes_nombre(m["autores"][0])[0], m["anio"], titulo, campos, usadas_bib))
    else:
        campos = {"author": bib_autores(m["autores"]), "journal": fuente, "volume": m.get("volumen"), "number": m.get("numero"), "pages": paginas_fmt(m.get("paginas") or "").replace("–", "--"), **enlace_}
        ENTRADAS_BIB.append(bib_entrada("article", partes_nombre(m["autores"][0])[0], m["anio"], titulo, campos, usadas_bib))
for doi, m in REFS_E2.items():
    anio, fuente = CONGRESO.get(doi, (m["anio"], m["fuente"]))
    comunes = {"author": bib_autores(m["autores"]), "doi": doi}
    apellido = partes_nombre(m["autores"][0])[0]
    if doi in CONGRESO or m.get("tipo") in ("proceedings-article", "conference-paper") or "Conference" in (fuente or "") or "Proceedings" in (fuente or ""):
        ENTRADAS_BIB.append(bib_entrada("inproceedings", apellido, anio, m["titulo"], {**comunes, "booktitle": fuente, "pages": paginas_fmt(m.get("paginas") or "").replace("–", "--")}, usadas_bib))
    elif m.get("tipo") == "book":
        ENTRADAS_BIB.append(bib_entrada("book", apellido, anio, m["titulo"], {**comunes, "publisher": "Wiley", "edition": "3" if doi == "10.1002/9781118548387" else None}, usadas_bib))
    elif m.get("tipo") == "preprint":
        ENTRADAS_BIB.append(bib_entrada("misc", apellido, anio, m["titulo"], {**comunes, "howpublished": fuente}, usadas_bib))
    else:
        ENTRADAS_BIB.append(bib_entrada("article", apellido, anio, m["titulo"], {**comunes, "journal": fuente, "volume": m.get("volumen"), "number": m.get("numero"),
                                                                              "pages": paginas_fmt(m.get("paginas") or "").replace("–", "--")}, usadas_bib))
# Referencias sin DOI de la lista anterior, transcritas con los mismos datos (solo iniciales donde la lista tiene iniciales)
SIN_DOI_BIB = [
    ("article", "Benavoli", 2017, "Time for a change: A tutorial for comparing multiple classifiers through Bayesian analysis", {"author": "Benavoli, A. and Corani, G. and Demšar, J. and Zaffalon, M.", "journal": "Journal of Machine Learning Research", "volume": 18, "number": 77, "pages": "1--36"}),
    ("book", "Cohen", 1988, "Statistical power analysis for the behavioral sciences", {"author": "Cohen, J.", "edition": "2", "publisher": "Lawrence Erlbaum Associates"}),
    ("article", "DeLong", 1988, "Comparing the areas under two or more correlated receiver operating characteristic curves: A nonparametric approach", {"author": "DeLong, E. R. and DeLong, D. M. and Clarke-Pearson, D. L.", "journal": "Biometrics", "volume": 44, "number": 3, "pages": "837--845"}),
    ("article", "Demšar", 2006, "Statistical comparisons of classifiers over multiple data sets", {"author": "Demšar, J.", "journal": "Journal of Machine Learning Research", "volume": 7, "pages": "1--30"}),
    ("book", "Efron", 1993, "An introduction to the bootstrap", {"author": "Efron, B. and Tibshirani, R. J.", "publisher": "Chapman & Hall"}),
    ("article", "Friedman", 1937, "The use of ranks to avoid the assumption of normality implicit in the analysis of variance", {"author": "Friedman, M.", "journal": "Journal of the American Statistical Association", "volume": 32, "number": 200, "pages": "675--701"}),
    ("article", "Geirhos", 2020, "Shortcut learning in deep neural networks", {"author": "Geirhos, R. and Jacobsen, J.-H. and Michaelis, C. and Zemel, R. and Brendel, W. and Bethge, M. and Wichmann, F. A.", "journal": "Nature Machine Intelligence", "volume": 2, "number": 11, "pages": "665--673"}),
    ("article", "Gretton", 2012, "A kernel two-sample test", {"author": "Gretton, A. and Borgwardt, K. M. and Rasch, M. J. and Schölkopf, B. and Smola, A.", "journal": "Journal of Machine Learning Research", "volume": 13, "pages": "723--773"}),
    ("article", "Holm", 1979, "A simple sequentially rejective multiple test procedure", {"author": "Holm, S.", "journal": "Scandinavian Journal of Statistics", "volume": 6, "number": 2, "pages": "65--70"}),
    ("inproceedings", "Kingma", 2015, "Adam: A method for stochastic optimization", {"author": "Kingma, D. P. and Ba, J.", "booktitle": "International Conference on Learning Representations", "url": "https://arxiv.org/abs/1412.6980"}),
    ("inproceedings", "Lopez-Paz", 2017, "Revisiting classifier two-sample tests", {"author": "Lopez-Paz, D. and Oquab, M.", "booktitle": "International Conference on Learning Representations"}),
    ("article", "McNemar", 1947, "Note on the sampling error of the difference between correlated proportions or percentages", {"author": "McNemar, Q.", "journal": "Psychometrika", "volume": 12, "number": 2, "pages": "153--157"}),
    ("article", "Nadeau", 2003, "Inference for the generalization error", {"author": "Nadeau, C. and Bengio, Y.", "journal": "Machine Learning", "volume": 52, "number": 3, "pages": "239--281"}),
    ("article", "Page", 2021, "The PRISMA 2020 statement: An updated guideline for reporting systematic reviews", {"author": "Page, M. J. and McKenzie, J. E. and Bossuyt, P. M. and others", "journal": "BMJ", "volume": 372, "pages": "n71", "doi": "10.1136/bmj.n71"}),
    ("article", "Schuirmann", 1987, "A comparison of the two one-sided tests procedure and the power approach for assessing the equivalence of average bioavailability", {"author": "Schuirmann, D. J.", "journal": "Journal of Pharmacokinetics and Biopharmaceutics", "volume": 15, "number": 6, "pages": "657--680"}),
    ("misc", "SCImago", 2026, "SJR: SCImago Journal & Country Rank", {"author": "{SCImago}", "howpublished": "Conjunto de datos", "url": "https://www.scimagojr.com"}),
    ("article", "Shapiro", 1965, "An analysis of variance test for normality (complete samples)", {"author": "Shapiro, S. S. and Wilk, M. B.", "journal": "Biometrika", "volume": 52, "number": "3/4", "pages": "591--611"}),
    ("article", "Wilcoxon", 1945, "Individual comparisons by ranking methods", {"author": "Wilcoxon, F.", "journal": "Biometrics Bulletin", "volume": 1, "number": 6, "pages": "80--83"}),
    ("misc", "Zeng", 2025, "UAV--IDS2025", {"author": "Zeng, Q.", "howpublished": "Código fuente, GitHub", "url": "https://github.com/QingliZeng/UAV--IDS2025"}),
    ("misc", "Zeng", 2025, "UAVIDS-2025", {"author": "Zeng, Q. and Bashir, A. and Nait-Abdesselam, F.", "howpublished": "Conjunto de datos, Zenodo", "doi": "10.5281/zenodo.15336998"}),
]
ENTRADAS_BIB += [bib_entrada(t, a, y_, ti, c, usadas_bib) for t, a, y_, ti, c in SIN_DOI_BIB]
assert len(ENTRADAS_BIB) == len(BIBLIOGRAFIA), f"BibTeX con {len(ENTRADAS_BIB)} entradas y bibliografía con {len(BIBLIOGRAFIA)}"
(LIT2 / "referencias.bib").write_text("% Bibliografía del Entregable 2 (UAVIDS-2025), generada por la sección 10 del cuaderno. Codificación UTF-8.\n\n"
                                      + "\n\n".join(sorted(ENTRADAS_BIB, key=lambda t: clave_orden(t.split("{", 1)[1]))) + "\n", encoding="utf-8")

- Abbas, S., Fayaz, M., Ullah, A., et al. (2026). A Federated Learning-Based Hybrid Architecture for Multi-Class Attack Detection in UAVs Communications. *IEEE Open Journal of the Communications Society, 7*, 6121–6135. <https://doi.org/10.1109/ojcoms.2026.3698792>
- Abdulhammed, R., Al-Dabag, M. L. A., Najeeb, S. M. M., et al. (2026). Assessing learning-based models for intrusion detection in uav networks. *Archives for Technical Sciences*, 229. <https://doi.org/10.70102/afts.2026.1836.229>
- Aburasain, R. Y. (2024). Enhanced Black Widow Optimization With Hybrid Deep Learning Enabled Intrusion Detection in Internet of Things-Based Smart Farming. *IEEE Access, 12*, 16621–16631. <https://doi.org/10.1109/access.2024.3359043>
- Aghaei, A. A. (2025). fKAN: Fractional Kolmogorov–Arnold Networks with trainable Jacobi basis functions. *Neurocomputing, 623*, 129414. <https://doi.org/10.1016/j.neucom.2025.129414>
- Alharthi, R. S. (2024). Enhancing unmanned aerial vehicle and smart grid communication security using a ConvLSTM model for intrusion detection. *Frontiers in Energy Research, 12*. <https://doi.org/10.3389/fenrg.2024.1491332>
- Alissa, K. A., Alotaibi, S. S., Alrayes, F. S., et al. (2022). Crystal Structure Optimization with Deep-Autoencoder-Based Intrusion Detection for Secure Internet of Drones Environment. *Drones, 6*(10), 297. <https://doi.org/10.3390/drones6100297>
- Alotaibi, S. S., Sayed, A. E., Elhameed, E. S. A., et al. (2024). Enhancing Security in IoT-Assisted UAV Networks Using Adaptive Mongoose Optimization Algorithm With Deep Learning. *IEEE Access, 12*, 63768–63776. <https://doi.org/10.1109/access.2024.3392618>
- Alpaydın, E. (1999). Combined 5 × 2 cv F Test for Comparing Supervised Classification Learning Algorithms. *Neural Computation, 11*(8), 1885–1892. <https://doi.org/10.1162/089976699300016007>
- Alruwaili, F. J. y Tunc, C. (2026). Federated Lightweight Intrusion Detection in Drone Swarms with Knowledge Distillation. *arXiv*. <https://doi.org/10.48550/arxiv.2607.17025>
- Alsaraireh, S., Al-Fayoumi, M. A. y Alnabhan, M. (2026). Dual-Impact Feature Selection for Adversarially Robust, Functionality-Preserving UAV Intrusion Detection. *Drones, 10*(7), 503. <https://doi.org/10.3390/drones10070503>
- Alshammari, S. M., Alganmi, N. A., Ba-Aoum, M. H., et al. (2024). Hybrid arithmetic optimization algorithm with deep learning model for secure Unmanned Aerial Vehicle networks. *AIMS Mathematics, 9*(3), 7131–7151. <https://doi.org/10.3934/math.2024348>
- Alshamrani, A. y Alghamdi, A. M. (2026). Zero-shot attack detection in UAV networks using foundation models. *Alexandria Engineering Journal, 136*, 105–124. <https://doi.org/10.1016/j.aej.2025.12.065>
- Angelopoulos, A. N. y Bates, S. (2023). Conformal Prediction: A Gentle Introduction. *Foundations and Trends® in Machine Learning, 16*(4), 494–591. <https://doi.org/10.1561/2200000101>
- Arık, S. Ö. y Pfister, T. (2021). TabNet: Attentive Interpretable Tabular Learning. En *Proceedings of the AAAI Conference on Artificial Intelligence*. <https://doi.org/10.1609/aaai.v35i8.16826>
- Arulalan, V., Balamurugan, G. y Premanand, V. (2023). A Deep CNN Framework for UAV Intrusion Detection in Intelligent Systems. *International Journal on Recent and Innovation Trends in Computing and Communication, 11*(9), 227–236. <https://doi.org/10.17762/ijritcc.v11i9.8338>
- Ashraf, S. N., Manickam, S., Zia, S. S., et al. (2023). IoT empowered smart cybersecurity framework for intrusion detection in internet of drones. *Scientific Reports, 13*(1), 18422. <https://doi.org/10.1038/s41598-023-45065-8>
- Attaullah, H. M., Basheer, S., Ehsan, M., et al. (2026a). Multi-Agent Federated Edge Learning for UAV-IDS in Smart City IoT Environment. *IEEE Transactions on Consumer Electronics, 72*(2), 3988–3995. <https://doi.org/10.1109/tce.2026.3667039>
- Attaullah, H. M., Ehsan, M., Basheer, S., et al. (2026b). GenFed-IDS: A Lightweight Federated Generative AI Framework for UAV Anomaly Detection in Rescue Operations. *IEEE Transactions on Consumer Electronics, 72*(3), 7662–7669. <https://doi.org/10.1109/tce.2026.3658881>
- Attaullah, H. M., Khan, I. U., Alam, M. M., et al. (2026c). BPBiLSTM-IDS: a lightweight intrusion detection framework for cyber-physical UAV networks. *Scientific Reports, 16*(1). <https://doi.org/10.1038/s41598-026-55446-4>
- Babu, S. A., Ranganath, A., Goswami, M. M., et al. (2024). Modified Marine Predators Algorithm With Deep Learning-Driven Security Solution for IoT-Assisted UAV Networks. *IEEE Access, 12*, 54991–54998. <https://doi.org/10.1109/access.2024.3386570>
- Bakiri, A., Fenanir, S. y Goudjil, L. (2026). H-MicroKAN: An energy-aware hierarchical Kolmogorov–Arnold Network for TinyML UAV intrusion detection with conditional inference. *Journal of Systems Architecture, 181*, 104038. <https://doi.org/10.1016/j.sysarc.2026.104038>
- Balgoon, J. O. (2026). Edge-Aware GATv2-BiLSTM Framework for Spatiotemporal Intrusion Detection in Autonomous UAV Command-and-Control Networks. *International Journal of Intelligent Systems and Applications in Engineering*. <https://ijisae.org/index.php/IJISAE/article/view/8549>
- Batur Şahin, C. (2026). Quantum-Resilient Federated Learning for Multi-Layer Cyber Anomaly Detection in UAV Systems. *Sensors, 26*(2), 509. <https://doi.org/10.3390/s26020509>
- Benavoli, A., Corani, G., Demšar, J. y Zaffalon, M. (2017). Time for a change: A tutorial for comparing multiple classifiers through Bayesian analysis. *Journal of Machine Learning Research, 18*(77), 1–36.
- Bengio, Y., Simard, P. y Frasconi, P. (1994). Learning long-term dependencies with gradient descent is difficult. *IEEE Transactions on Neural Networks, 5*(2), 157–166. <https://doi.org/10.1109/72.279181>
- Biswas, T. K., Zannat, A., Ishtiaq, W., et al. (2026). A novel unified lightweight temporal-spatial transformer approach for intrusion detection in drone networks. *Scientific Reports, 16*(1). <https://doi.org/10.1038/s41598-026-45063-6>
- Brown, M. B. y Forsythe, A. B. (1974). Robust Tests for the Equality of Variances. *Journal of the American Statistical Association, 69*(346), 364–367. <https://doi.org/10.1080/01621459.1974.10482955>
- Cengiz, K., Lipsa, S., Dash, R. K., et al. (2024). A Novel Intrusion Detection System Based on Artificial Neural Network and Genetic Algorithm With a New Dimensionality Reduction Technique for UAV Communication. *IEEE Access, 12*, 4925–4937. <https://doi.org/10.1109/access.2024.3349469>
- Cho, K., van Merriënboer, B., Gülçehre, Ç., et al. (2014). Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation. En *Conference on Empirical Methods in Natural Language Processing (EMNLP)*. <https://doi.org/10.3115/v1/d14-1179>
- Cohen, J. (1988). *Statistical power analysis for the behavioral sciences* (2.ª ed.). Lawrence Erlbaum Associates.
- Cover, T. M. y Hart, P. E. (1967). Nearest neighbor pattern classification. *IEEE Transactions on Information Theory, 13*(1), 21–27. <https://doi.org/10.1109/tit.1967.1053964>
- DeLong, E. R., DeLong, D. M. y Clarke-Pearson, D. L. (1988). Comparing the areas under two or more correlated receiver operating characteristic curves: A nonparametric approach. *Biometrics, 44*(3), 837–845.
- Demšar, J. (2006). Statistical comparisons of classifiers over multiple data sets. *Journal of Machine Learning Research, 7*, 1–30.
- Di, Q., Du, K., Chen, G., et al. (2026). Characterizing Robustness of Tabular Intrusion Detection System Models Under Sensor Degradation: A Systematic Empirical Evaluation for Unmanned Aerial Vehicle Ad Hoc Networks. *Concurrency and Computation Practice and Experience, 38*(22). <https://doi.org/10.1002/cpe.71022>
- Donahue, J., Hendricks, L. A., Guadarrama, S., et al. (2015). Long-term recurrent convolutional networks for visual recognition and description. En *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)*. <https://doi.org/10.1109/cvpr.2015.7298878>
- Efron, B. y Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall.
- Elman, J. L. (1990). Finding Structure in Time. *Cognitive Science, 14*(2), 179–211. <https://doi.org/10.1207/s15516709cog1402_1>
- Friedman, M. (1937). The use of ranks to avoid the assumption of normality implicit in the analysis of variance. *Journal of the American Statistical Association, 32*(200), 675–701.
- Gal, Y. y Ghahramani, Z. (2016). Dropout as a Bayesian Approximation: Representing Model Uncertainty in Deep Learning. En *International Conference on Machine Learning*. <https://doi.org/10.48550/arxiv.1506.02142>
- Geirhos, R., Jacobsen, J.-H., Michaelis, C., Zemel, R., Brendel, W., Bethge, M. y Wichmann, F. A. (2020). Shortcut learning in deep neural networks. *Nature Machine Intelligence, 2*(11), 665–673.
- Gharami, K. y Moni, S. S. (2025). An Efficient Privacy-preserving Intrusion Detection Scheme for UAV Swarm Networks. En *IEEE/AIAA Digital Avionics Systems Conference (DASC)* (pp. 1–10). <https://doi.org/10.1109/dasc66011.2025.11257309>
- Goodfellow, I., Pouget-Abadie, J., Mirza, M., et al. (2020). Generative adversarial networks. *Communications of the ACM, 63*(11), 139–144. <https://doi.org/10.1145/3422622>
- Gorishniy, Y., Rubachev, I., Khrulkov, V., et al. (2021). Revisiting Deep Learning Models for Tabular Data. En *Advances in Neural Information Processing Systems 34*. <https://doi.org/10.48550/arxiv.2106.11959>
- Graves, A. y Schmidhuber, J. (2005). Framewise phoneme classification with bidirectional LSTM and other neural network architectures. *Neural Networks, 18*(5-6), 602–610. <https://doi.org/10.1016/j.neunet.2005.06.042>
- Greenhouse, S. W. y Geisser, S. (1959). On Methods in the Analysis of Profile Data. *Psychometrika, 24*(2), 95–112. <https://doi.org/10.1007/bf02289823>
- Gretton, A., Borgwardt, K. M., Rasch, M. J., Schölkopf, B. y Smola, A. (2012). A kernel two-sample test. *Journal of Machine Learning Research, 13*, 723–773.
- Guo, C., Pleiss, G., Sun, Y., et al. (2017). On Calibration of Modern Neural Networks. En *International Conference on Machine Learning*. <https://doi.org/10.48550/arxiv.1706.04599>
- Hadi, H. J., Cao, Y., Li, S., et al. (2024). Real-time fusion multi-tier DNN-based collaborative IDPS with complementary features for secure UAV-enabled 6G networks. *Expert Systems with Applications, 252*, 124215. <https://doi.org/10.1016/j.eswa.2024.124215>
- Hadi, H. J., Khan, M. K. y Ahmad, N. (2026). A Real-Time Multi-Tier Machine Learning Intrusion Detection Framework for Securing Ground Control Station–UAV Communications. *IEEE Open Journal of the Communications Society, 7*, 3881–3900. <https://doi.org/10.1109/ojcoms.2026.3683883>
- Hakeem, A., Sabir, M., Alhebshi, R. M., et al. (2025). Integrating artificial intelligence for improved security of IoT-drones through cyber-physical attack detection. *Frontiers in Computer Science, 7*. <https://doi.org/10.3389/fcomp.2025.1545282>
- Hamad, A. H., Hussein, N. K. y Abdulghani, A. M. (2025). A Deep Learning Paradigm for Intrusion Detection in Unmanned Aerial Vehicle Networks Using Extended LSTM. *International Journal of Intelligent Engineering and Systems, 18*(4), 507–523. <https://doi.org/10.22266/ijies2025.0531.33>
- Hansen, P. R., Lunde, A. y Nason, J. M. (2011). The Model Confidence Set. *Econometrica, 79*(2), 453–497. <https://doi.org/10.3982/ecta5771>
- He, K., Fan, H., Wu, Y., et al. (2020). Momentum Contrast for Unsupervised Visual Representation Learning. En *Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)*. <https://doi.org/10.1109/cvpr42600.2020.00975>
- He, K., Zhang, X., Ren, S., et al. (2016). Deep Residual Learning for Image Recognition. En *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)*. <https://doi.org/10.1109/cvpr.2016.90>
- Hinton, G. E. y Salakhutdinov, R. (2006). Reducing the Dimensionality of Data with Neural Networks. *Science, 313*(5786), 504–507. <https://doi.org/10.1126/science.1127647>
- Hinton, G. E., Osindero, S. y Teh, Y.-W. (2006). A Fast Learning Algorithm for Deep Belief Nets. *Neural Computation, 18*(7), 1527–1554. <https://doi.org/10.1162/neco.2006.18.7.1527>
- Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics, 6*(2), 65–70.
- Hosmer, D. W., Lemeshow, S. A. y Sturdivant, R. X. (2013). *Applied Logistic Regression* (3.ª ed.). Wiley. <https://doi.org/10.1002/9781118548387>
- Iman, R. L. y Davenport, J. M. (1980). Approximations of the critical region of the Friedman statistic. *Communications in Statistics - Theory and Methods, 9*(6), 571–595. <https://doi.org/10.1080/03610928008827904>
- Junaid, Algarni, F., Khan, M. I., et al. (2026). An AI-driven framework for intrusion detection and predictive threat modeling in UAV ecosystem. *Internet of Things, 40*, 102095. <https://doi.org/10.1016/j.iot.2026.102095>
- Kalantidis, Y., Sarıyıldız, M. B., Pion, N., et al. (2020). Hard Negative Mixing for Contrastive Learning. En *Advances in Neural Information Processing Systems 33*. <https://doi.org/10.48550/arxiv.2010.01028>
- Karani, N., Erdil, E., Chaitanya, K., et al. (2020). Test-time adaptable neural networks for robust medical image segmentation. *Medical Image Analysis, 68*, 101907. <https://doi.org/10.1016/j.media.2020.101907>
- Katbar, N. M., Hakeem, A., Ahmed, I., et al. (2024). Radial Basis Function Neural Networks (RBFNN) for Secure Intrusion Detection in the Internet of Drones (IoD). *Proceedings of the Bulgarian Academy of Sciences, 77*(8). <https://doi.org/10.7546/crabs.2024.08.09>
- Katib, I. A., Ragab, M., Sharaf, S. A., et al. (2025). Integrating convolutional stacked long short-term memory with bio-inspired algorithm for secure unmanned aerial vehicle communication in SDN environment. *Results in Engineering, 27*, 106838. <https://doi.org/10.1016/j.rineng.2025.106838>
- Khosla, P., Teterwak, P., Wang, C., et al. (2020). Supervised Contrastive Learning. En *Advances in Neural Information Processing Systems 33*. <https://doi.org/10.48550/arxiv.2004.11362>
- Kingma, D. P. y Ba, J. (2015). Adam: A method for stochastic optimization. En *International Conference on Learning Representations*. <https://arxiv.org/abs/1412.6980>
- Lakshminarayanan, B., Pritzel, A. y Blundell, C. (2017). Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles. En *Advances in Neural Information Processing Systems 30*. <https://doi.org/10.48550/arxiv.1612.01474>
- Li, M., Jin, Y., Ji, W., et al. (2026a). Enhancing UAV Intrusion Detection Against Adversarial Samples Based on AdvGAN and Decision Boundary Reconstruction. *IEEE Internet of Things Journal, 13*(10), 20705–20717. <https://doi.org/10.1109/jiot.2026.3665688>
- Li, X., Zhang, T., Wang, J., et al. (2026b). Enhance UAV Network Resilience by Malicious Traffic Detection: A Twin Graph Encoder Approach. *IEEE Transactions on Communications, 74*, 9138–9153. <https://doi.org/10.1109/tcomm.2026.3693173>
- Liu, Z., Wang, Y., Vaidya, S., et al. (2024). KAN: Kolmogorov-Arnold Networks. *arXiv*. <https://doi.org/10.48550/arxiv.2404.19756>
- Lopez-Paz, D. y Oquab, M. (2017). Revisiting classifier two-sample tests. En *International Conference on Learning Representations*.
- Mauchly, J. W. (1940). Significance Test for Sphericity of a Normal n-Variate Distribution. *The Annals of Mathematical Statistics, 11*(2), 204–209. <https://doi.org/10.1214/aoms/1177731915>
- McNemar, Q. (1947). Note on the sampling error of the difference between correlated proportions or percentages. *Psychometrika, 12*(2), 153–157.
- Menssouri, S., Delamou, M., Ibrahimi, K., et al. (2024). Enhanced Intrusion Detection System for Multiclass Classification in UAV Networks. En *IEEE International Symposium on Personal, Indoor and Mobile Radio Communications (PIMRC)* (pp. 1–6). <https://doi.org/10.1109/pimrc59610.2024.10817244>
- Mohamed, M., Mohamed, D. y Alosman, K. (2026). Spatio-temporal graph neural networks for zero-day intrusion detection in low-altitude autonomous UAV swarms. *Ad Hoc Networks, 191*, 104251. <https://doi.org/10.1016/j.adhoc.2026.104251>
- Nadeau, C. y Bengio, Y. (2003). Inference for the generalization error. *Machine Learning, 52*(3), 239–281.
- Negm, N., Alamro, H., Allafi, R., et al. (2024). Tasmanian devil optimization with deep autoencoder for intrusion detection in IoT assisted unmanned aerial vehicle networks. *Ain Shams Engineering Journal, 15*(11), 102943. <https://doi.org/10.1016/j.asej.2024.102943>
- Northcutt, C. G., Jiang, L. y Chuang, I. L. (2021). Confident Learning: Estimating Uncertainty in Dataset Labels. *Journal of Artificial Intelligence Research, 70*, 1373–1411. <https://doi.org/10.1613/jair.1.12125>
- Ozsoy, D. B., Ozcan, A. y Al-Hubaishi, M. (2026). Federated Learning for Intrusion Detection in UAV Networks. *Turkish Journal of Mathematics and Computer Science, 18*(2), 481–503. <https://doi.org/10.47000/tjmcs.1828534>
- Page, M. J., McKenzie, J. E., Bossuyt, P. M., et al. (2021). The PRISMA 2020 statement: An updated guideline for reporting systematic reviews. *BMJ, 372*, n71. <https://doi.org/10.1136/bmj.n71>
- Qezelbash-Chamak, J., Hicklin, K. y Kim, M. (2025). KANBalance: Kolmogorov–Arnold network mitigates class imbalance. *Pattern Recognition, 171*, 112325. <https://doi.org/10.1016/j.patcog.2025.112325>
- Rousseeuw, P. J. y Van Driessen, K. (1999). A Fast Algorithm for the Minimum Covariance Determinant Estimator. *Technometrics, 41*(3), 212–223. <https://doi.org/10.1080/00401706.1999.10485670>
- Scarselli, F., Gori, M., Tsoi, A. C., et al. (2008). The Graph Neural Network Model. *IEEE Transactions on Neural Networks, 20*(1), 61–80. <https://doi.org/10.1109/tnn.2008.2005605>
- Schuirmann, D. J. (1987). A comparison of the two one-sided tests procedure and the power approach for assessing the equivalence of average bioavailability. *Journal of Pharmacokinetics and Biopharmaceutics, 15*(6), 657–680.
- SCImago (2026). *SJR: SCImago Journal & Country Rank* [Conjunto de datos]. <https://www.scimagojr.com>
- Shapiro, S. S. y Wilk, M. B. (1965). An analysis of variance test for normality (complete samples). *Biometrika, 52*(3/4), 591–611.
- Spiegelhalter, D. J. (1986). Probabilistic prediction in patient management and clinical trials. *Statistics in Medicine, 5*(5), 421–433. <https://doi.org/10.1002/sim.4780050506>
- Sundararajan, M., Taly, A. y Yan, Q. (2017). Axiomatic Attribution for Deep Networks. En *International Conference on Machine Learning*. <https://doi.org/10.48550/arxiv.1703.01365>
- SureshBabu, P. y Geetha, M. (2026). Deep Reinforced Adaptive Graph Optimization Network for Intelligent Intrusion Detection in UAV Communication Networks. *Indian Engineering Journal*. <https://indianengineeringjournal.com/index.php/journal/article/view/270>
- Tan, X., Su, S., Zuo, Z., et al. (2019). Intrusion Detection of UAVs Based on the Deep Belief Network Optimized by PSO. *Sensors, 19*(24), 5529. <https://doi.org/10.3390/s19245529>
- Tasfia, R., Chennamaneni, A. y Nag, A. K. (2026). Revisiting adversarial robustness in NIDS: a multi-dataset evaluation of test-time evasion attacks. *International Journal of Information Security, 25*, 184. <https://doi.org/10.1007/s10207-026-01353-8>
- Vaswani, A., Shazeer, N., Parmar, N., et al. (2017). Attention Is All You Need. En *Advances in Neural Information Processing Systems 30*. <https://doi.org/10.48550/arxiv.1706.03762>
- Vickers, A. J. y Elkin, E. B. (2006). Decision Curve Analysis: A Novel Method for Evaluating Prediction Models. *Medical Decision Making, 26*(6), 565–574. <https://doi.org/10.1177/0272989x06295361>
- Wang, D., Shelhamer, E., Liu, S., et al. (2020). Tent: Fully Test-time Adaptation by Entropy Minimization. *arXiv*. <https://doi.org/10.48550/arxiv.2006.10726>
- Wang, J., Alsubait, T., Mufleh, A. S. S., et al. (2026). Residual-aware lightweight deep learning framework for high-fidelity intrusion detection in UAV swarm networks. *Scientific Reports, 16*(1). <https://doi.org/10.1038/s41598-026-52524-5>
- Wilcoxon, F. (1945). Individual comparisons by ranking methods. *Biometrics Bulletin, 1*(6), 80–83.
- Wu, Y., Yang, L., Zhang, L., et al. (2024). Intrusion Detection for Unmanned Aerial Vehicles Security: A Tiny Machine Learning Model. *IEEE Internet of Things Journal, 11*(12), 20970–20982. <https://doi.org/10.1109/jiot.2024.3360231>
- Zarkadis, I.-C. y Douligeris, C. (2026). XAI and Statistical Analysis for Reliable Intrusion Detection in the UAVIDS-2025 Dataset: From Tree to Hybrid and Tabular DNN Ensembles. En *International Conference on Computer, Information and Telecommunication Systems* (pp. 1–8). <https://doi.org/10.1109/cits70307.2026.11637173>
- Zeng, Q. (2025). *UAV--IDS2025* [Código fuente]. GitHub. <https://github.com/QingliZeng/UAV--IDS2025>
- Zeng, Q., Bashir, A. y Nait-Abdesselam, F. (2025). *UAVIDS-2025* [Conjunto de datos]. Zenodo. <https://doi.org/10.5281/zenodo.15336998>
- Zeng, Q., Bashir, A. y Naït-Abdesselam, F. (2025). UAVIDS-2025: A Benchmark Dataset for Intrusion Detection in UAV Networks Using Machine Learning Techniques. En *IEEE Conference on Communications and Network Security* (pp. 1–9). <https://doi.org/10.1109/cns66487.2025.11194990>
- Zhou, K., Yang, Y., Qiao, Y., et al. (2023). MixStyle Neural Networks for Domain Generalization and Adaptation. *International Journal of Computer Vision, 132*(3), 822–836. <https://doi.org/10.1007/s11263-023-01913-8>
- Zhou, W., Li, B. y Zeng, X. (2026). Intrusion detection model of UAV system based on machine learning and neural network. *Frontiers in Physics, 13*. <https://doi.org/10.3389/fphy.2025.1660104>
- Zoph, B., Bello, I., Kumar, S., et al. (2022). ST-MoE: Designing Stable and Transferable Sparse Expert Models. *arXiv*. <https://doi.org/10.48550/arxiv.2202.08906>

35712